In [1]:
pip install pandas numpy faker

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   --------------- ------------------------ 0.8/2.0 MB 5.4 MB/s eta 0:00:01
   ------------------------------------ --- 1.8/2.0 MB 4.4 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 4.2 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

In [2]:
fake = Faker()

random.seed(42)
np.random.seed(42)

In [15]:
customers_df.to_csv("customers.csv", index=False)
products_df.to_csv("products.csv", index=False)
orders_df.to_csv("orders.csv", index=False)

In [3]:
customers = []

cities = [
    "Mumbai",
    "Delhi",
    "Bangalore",
    "Pune",
    "Hyderabad",
    "Chennai"
]

for i in range(1, 201):

    customers.append({
        "customer_id": f"C{i:03}",
        "age": random.randint(21, 60),
        "city": random.choice(cities),
        "signup_date": fake.date_between(
            start_date='-2y',
            end_date='today'
        )
    })

customers_df = pd.DataFrame(customers)



In [5]:
print(customers_df.head())

    customer_id  age       city signup_date
0          C001   28     Mumbai  2024-12-19
1          C002   38      Delhi  2026-04-24
2          C003   35      Delhi  2025-09-04
3          C004   27    Chennai  2026-01-24
4          C005   55     Mumbai  2025-04-04
..          ...  ...        ...         ...
195        C196   37  Hyderabad  2025-02-21
196        C197   52  Bangalore  2025-08-24
197        C198   24     Mumbai  2026-01-11
198        C199   48  Bangalore  2025-11-30
199        C200   23     Mumbai  2024-11-09

[200 rows x 4 columns]


In [6]:
accounts = []

account_types = [
    "Savings",
    "Credit",
    "Business"
]

for i in range(1, 201):

    accounts.append({
        "account_id": f"A{i:03}",
        "customer_id": f"C{i:03}",
        "account_type": random.choices(
            account_types,
            weights=[70, 20, 10]
        )[0]
    })

accounts_df = pd.DataFrame(accounts)

print(accounts_df.head())

  account_id customer_id account_type
0       A001        C001      Savings
1       A002        C002      Savings
2       A003        C003     Business
3       A004        C004      Savings
4       A005        C005      Savings


In [13]:
# ==========================================
# TRANSACTIONS TABLE GENERATION
# ==========================================

transactions = []

transaction_id = 1

merchant_categories = {
    "Amazon": "Shopping",
    "Flipkart": "Shopping",
    "Swiggy": "Food",
    "Zomato": "Food",
    "Netflix": "Entertainment",
    "Uber": "Transport",
    "BigBasket": "Groceries",
    "Croma": "Electronics"
}

merchants = list(merchant_categories.keys())

date_range = pd.date_range(
    start="2023-01-01",
    end="2024-12-31"
)

high_value_customers = random.sample(
    accounts_df["account_id"].tolist(),
    20
)

dormant_customers = random.sample(
    accounts_df["account_id"].tolist(),
    40
)

account_balances = {
    account: random.randint(10000, 50000)
    for account in accounts_df["account_id"]
}

# ==========================================
# GENERATE TRANSACTIONS
# ==========================================

for date in date_range:

    daily_transactions = random.randint(5, 15)

    for _ in range(daily_transactions):

        # Random account
        account = random.choice(
            accounts_df["account_id"].tolist()
        )

        # Dormant customer logic
        if (
            account in dormant_customers
            and date > pd.Timestamp("2024-06-30")
        ):
            continue

        # Salary credits in first 5 days
        if date.day <= 5:

            transaction_type = "credit"

            amount = random.randint(
                30000,
                120000
            )

            merchant = "Salary Deposit"

            category = "Income"

        else:

            transaction_type = random.choices(
                ["debit", "credit"],
                weights=[80, 20]
            )[0]

            weekend = date.weekday() >= 5

            if weekend:

                amount = random.randint(
                    1000,
                    15000
                )

            else:

                amount = random.randint(
                    200,
                    8000
                )

            merchant = random.choice(merchants)

            category = merchant_categories[merchant]

        # High-value customer logic
        if account in high_value_customers:

            amount *= 3

        # Fraud logic
        fraud_flag = 0

        if random.random() < 0.02:

            fraud_flag = 1

            amount = random.randint(
                50000,
                200000
            )

            merchant = "Suspicious Transfer"

            category = "Fraud"

        # Transaction status
        transaction_status = random.choices(
            ["Success", "Failed", "Pending"],
            weights=[90, 5, 5]
        )[0]

        # Random time
        hour = random.randint(0, 23)

        minute = random.randint(0, 59)

        transaction_datetime = (
            pd.Timestamp(date)
            + pd.Timedelta(
                hours=hour,
                minutes=minute
            )
        )

        # Running balance
        current_balance = account_balances[account]

        if transaction_type == "credit":

            current_balance += amount

        else:

            current_balance -= amount

        account_balances[account] = current_balance

        # Save transaction
        transactions.append({

            "transaction_id": transaction_id,

            "account_id": account,

            "transaction_date": transaction_datetime,

            "transaction_type": transaction_type,

            "merchant": merchant,

            "transaction_category": category,

            "amount": amount,

            "transaction_status": transaction_status,

            "fraud_flag": fraud_flag,

            "running_balance": current_balance
        })

        transaction_id += 1

# ==========================================
# CREATE DATAFRAME
# ==========================================

transactions_df = pd.DataFrame(transactions)

print(transactions_df)

# ==========================================


   transaction_id account_id    transaction_date transaction_type  \
0               1       A017 2023-01-01 11:37:00           credit   
1               2       A052 2023-01-01 01:36:00           credit   
2               3       A057 2023-01-01 12:21:00           credit   
3               4       A062 2023-01-01 09:09:00           credit   
4               5       A037 2023-01-01 17:40:00           credit   

         merchant transaction_category  amount transaction_status  fraud_flag  \
0  Salary Deposit               Income   34596            Success           0   
1  Salary Deposit               Income   64222            Success           0   
2  Salary Deposit               Income  257859            Success           0   
3  Salary Deposit               Income   81885            Success           0   
4  Salary Deposit               Income   99389            Success           0   

   running_balance  
0            68026  
1            82127  
2           283678  
3            9

In [14]:
print(transactions_df)

      transaction_id account_id    transaction_date transaction_type  \
0                  1       A017 2023-01-01 11:37:00           credit   
1                  2       A052 2023-01-01 01:36:00           credit   
2                  3       A057 2023-01-01 12:21:00           credit   
3                  4       A062 2023-01-01 09:09:00           credit   
4                  5       A037 2023-01-01 17:40:00           credit   
...              ...        ...                 ...              ...   
6834            6835       A045 2024-12-31 05:44:00            debit   
6835            6836       A145 2024-12-31 13:14:00            debit   
6836            6837       A173 2024-12-31 23:52:00            debit   
6837            6838       A194 2024-12-31 18:47:00            debit   
6838            6839       A050 2024-12-31 12:16:00            debit   

            merchant transaction_category  amount transaction_status  \
0     Salary Deposit               Income   34596            Su

In [15]:
transactions_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6839 entries, 0 to 6838
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   transaction_id        6839 non-null   int64         
 1   account_id            6839 non-null   str           
 2   transaction_date      6839 non-null   datetime64[us]
 3   transaction_type      6839 non-null   str           
 4   merchant              6839 non-null   str           
 5   transaction_category  6839 non-null   str           
 6   amount                6839 non-null   int64         
 7   transaction_status    6839 non-null   str           
 8   fraud_flag            6839 non-null   int64         
 9   running_balance       6839 non-null   int64         
dtypes: datetime64[us](1), int64(4), str(5)
memory usage: 534.4 KB


In [17]:
transactions_df.describe()

,transaction_id,transaction_date,amount,fraud_flag,running_balance
count,6839.000000,6839,6839.000000,6839.000000,6.839000e+03
mean,3420.000000,2023-12-16 14:08:06.272846,20840.712092,0.019886,1.978407e+05
min,1.000000,2023-01-01 00:11:00,200.000000,0.000000,-3.242010e+05
25%,1710.500000,2023-06-21 23:56:00,3285.000000,0.000000,5.385950e+04
50%,3420.000000,2023-12-12 04:30:00,6192.000000,0.000000,1.586010e+05
75%,5129.500000,2024-06-07 19:04:00,12857.500000,0.000000,2.916655e+05
max,6839.000000,2024-12-31 23:52:00,357810.000000,1.000000,1.638629e+06
std,1974.393578,NaN,39456.240283,0.139619,2.170548e+05


In [18]:
transactions_df["fraud_flag"].value_counts()

fraud_flag
0    6703
1     136
Name: count, dtype: int64

In [20]:
customers_df.to_csv("customers.csv", index=False)
accounts_df.to_csv("accounts.csv", index=False)
transactions_df.to_csv("transactions.csv", index=False)